Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS author ensemble vs Lag4: checkpoint recovery

Resume verified v2 CUDA checkpoints; compute missing evaluations and fits only. Same models,folds and hyperparameters. Exact serialized parameters and labels; FP32 probability replay tolerance1e-4. Full parent/result lineage saved.

In [ ]:
import os,sys,pathlib,json,hashlib,zipfile,io,base64,subprocess
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",PYTHONUNBUFFERED="1",MPLCONFIGDIR="/tmp/ensemble-mpl")
import torch
assert torch.cuda.is_available(),"Enable requested Kaggle GPU"
print("GPU",torch.cuda.get_device_name(0),flush=True)
project=pathlib.Path("/kaggle/working/riemann-project");project.mkdir(exist_ok=True)
sources=[p for p in pathlib.Path("/kaggle/input").rglob("source-integrity.json") if "proteus-lag4-classifier-source" in str(p)];assert len(sources)==1
raw=sources[0].read_bytes()
assert hashlib.sha256(raw).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(raw).items():
    p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
    data=(sources[0].parent/rel).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
    dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
payload=base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text())
assert hashlib.sha256(payload).hexdigest()=="cd9eb56ff9f61e147e81a37264b2aa22b6235cfe6db8faff533f383fb7d16d5f"
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    manifest=json.loads(z.read("ensemble-manifest.json"))
    for rel,digest in manifest.items():
        p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
        data=z.read(rel);assert hashlib.sha256(data).hexdigest()==digest
        dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
repro=project/"reproduction";experiment=repro/"ensemble_comparison"
inputs=[p for p in pathlib.Path("/kaggle/input").rglob("input-integrity.json") if "proteus-lag4-classifier-inputs" in str(p)];assert len(inputs)==1
input_manifest=json.loads(inputs[0].read_text());input_dir=inputs[0].parent
for name,digest in input_manifest["files_sha256"].items():
    h=hashlib.sha256()
    with (input_dir/name).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    assert h.hexdigest()==digest,name
(repro/"data").mkdir(exist_ok=True)
for split in ["train","val","test"]:
    (repro/"data"/f"proteus_{split}.npz").symlink_to(input_dir/f"proteus_{split}.npz")
(repro/"temporal_spd_research/cache").symlink_to(input_dir,target_is_directory=True)
envdir=pathlib.Path("/tmp/ensemble-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--quiet",*pins],check=True)
out=pathlib.Path("/kaggle/working/ensemble-results");out.mkdir(exist_ok=True)
env=dict(os.environ,ENSEMBLE_OUTPUT=str(out))
(out/"bundle-provenance.json").write_text(json.dumps(dict(ensemble_bundle_sha256="cd9eb56ff9f61e147e81a37264b2aa22b6235cfe6db8faff533f383fb7d16d5f",original_source_manifest_sha256=hashlib.sha256(raw).hexdigest(),input_manifest=input_manifest,source_manifest=manifest),indent=2))
print("Verified author ensemble and exact submitted Lag4. ShallowConvNet trains on CUDA.",flush=True)

print("CUDA DEVICES",torch.cuda.device_count(),flush=True)

receipts=list(pathlib.Path('/kaggle/input').rglob('checkpoint-integrity.json'));assert len(receipts)==1
assert hashlib.sha256(receipts[0].read_bytes()).hexdigest()=="c94ccfa0c183b0bc6c6fd76d413f62d26dd3b4a9cbb947fe167b542487ca1253"
checkpoint_receipt=json.loads(receipts[0].read_text())
parents=list(receipts[0].parent.rglob('ensemble-results/manifest.json'));assert len(parents)==1
parent_dir=parents[0].parent
for name,digest in checkpoint_receipt['files'].items():
    rel=pathlib.PurePosixPath(name);assert not rel.is_absolute() and '..' not in rel.parts
    assert hashlib.sha256((parent_dir/name).read_bytes()).hexdigest()==digest,name
env['ENSEMBLE_PARENT']=str(parent_dir)
env['ENSEMBLE_PARENT_MANIFEST_SHA256']="ab3ab25c19fa33bc61d81ac3b3fc1bdb51b2508ae4150aaa5bb751f896029792"
print('Verified pinned parent checkpoints for6folds',flush=True)


## Verify splits,stream state,serialization and CUDA training

In [ ]:
subprocess.run([python,"-m","pytest","-q","test_runner.py","test_analysis.py","test_replay.py","--tb=short"],cwd=experiment,env=env,check=True)

## Run fixed models; freeze CV before public test

In [ ]:
process=subprocess.Popen([python,"-u",str(experiment/"resume.py"),"--jobs","2"],cwd=experiment,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end="",flush=True)
code=process.wait()
if code:raise RuntimeError(f"Experiment exited with status {code};partial results are saved")


## Independently verified comparison

In [ ]:
print((out/"REPORT.md").read_text())
print("Artifacts:",sorted(p.name for p in out.iterdir()))